[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/11_LangGraph/capstone_agentic_rag/self_correcting_rag.ipynb)


# Capstone — Self-Correcting Agentic RAG

**Session 11, notebook 3.** Run top to bottom on your own.

Notebook 01 gave you a graph that can **loop** and **branch**. Notebook 02
gave you a **pause** you type into. This notebook puts both on a real product:
the RAG pipeline you already shipped in Module 10.

## The one problem this notebook solves

Module 10's chatbot always does the same four steps, in the same order:

```
search → rerank → guardrail → answer
```

If search is weak, it can only refuse. If the answer is ungrounded, it can only
hope the guardrail caught it. If the question is "What is the capital of France?"
asked of a shipping report, it cannot pause and ask a human.

That is not a retrieval problem. It is a **straight line** problem.

## What we do *not* rebuild

`HybridIndex`, `Reranker`, `load_document`, `chunk_documents`, `format_sources`,
`ANSWER_PROMPT`, and `CONDENSE_PROMPT` are **imported** from
`10_RAG/notebooks/production_rag_chatbot/rag_pipeline.py`. Chunking, BM25 + dense
fusion, RRF, and cross-encoder reranking are already taught. We wrap them.

| Module 10 chain | This graph |
|---|---|
| retrieve → rerank → generate, always | retrieve → **grade**, then generate only if sufficient |
| weak retrieval → refuse | weak retrieval → **rewrite** and try again (capped) |
| trust the generated text | **check groundedness**, regenerate once if needed |
| "I don't know" is the only exit | retries exhausted → **`interrupt()`**, a human decides |
| memory = a Python list on the class | memory = a **checkpointer**, same `thread_id` |

## How to self-study

1. Run one step. Read **What to look at**. Then go on.
2. After `compile()`, we draw the graph. After every turn, we print which nodes
   fired. Do not imagine the path — read the trace.
3. Grading and rewriting are live LLM calls. The *exact* retry path can change
   between runs. That is a property of agentic systems, not a bug.
4. **Turn 3 waits for you** if the graph escalates. Type an answer, or press
   Enter to refuse — same idea as notebook 02's yes/no.

## Roadmap

| Step | What you will see |
|---|---|
| 1 | Why the Module 10 chain cannot retry |
| 2 | Import the pipeline. Index the same PDF. |
| 3 | The whiteboard — every field, in English |
| 4 | Reused nodes: condense + retrieve |
| 5 | New loop: grade → rewrite → retrieve again |
| 6 | New loop: generate → groundedness → regenerate |
| 7 | Escalation: the pause from notebook 02 |
| 8 | Wire the graph and draw it |
| 9 | Four turns on one thread |
| 10 | The production module, tests, and the app |


---
# Setup

Quote the version pins — zsh reads `>=0.6` as a redirect (`zsh: 0.6 not found`).
After a fresh install: **Kernel → Restart Kernel**, then continue.


In [ ]:
%pip install -q "langgraph>=0.6" "langchain>=1.0" langchain-openai langchain-community \
    langchain-text-splitters langchain-pymupdf4llm pypdf docx2txt \
    sentence-transformers bm25s PyStemmer chromadb python-dotenv "numpy<2"

import importlib.metadata
print("langgraph :", importlib.metadata.version("langgraph"))
print("langchain :", importlib.metadata.version("langchain"))


In [ ]:
import warnings, os, sys, logging
warnings.filterwarnings("ignore")
for _n in ("httpx", "openai", "httpcore", "sentence_transformers", "transformers", "chromadb"):
    logging.getLogger(_n).setLevel(logging.ERROR)

from pathlib import Path
from dotenv import load_dotenv

# 👇 Same walk-up as notebooks 01 and 02. Keys live in GenAI-2026/.env or 10_RAG/.env,
# not always in this folder. First file wins for each key (override=False).
here = Path.cwd().resolve()
env_files = []
for folder in [here, *here.parents]:
    for candidate in (folder / ".env", folder / "10_RAG" / ".env"):
        if candidate.is_file() and candidate not in env_files:
            env_files.append(candidate)
            load_dotenv(candidate, override=False)

print("cwd            :", here)
print("Loaded .env    :", [str(p) for p in env_files] or "NONE FOUND")
print("OPENAI_API_KEY :", "set" if os.getenv("OPENAI_API_KEY") else "MISSING")


In [ ]:
def show_graph(compiled):
    print(compiled.get_graph().draw_mermaid())
    try:
        from IPython.display import Image, display
        display(Image(compiled.get_graph().draw_mermaid_png()))
    except Exception as e:
        print("PNG skipped (no mermaid.ink). The text diagram is enough.", e)


---
# Step 1 of 10 — Why the Module 10 chain cannot retry

**In one sentence:** a function that always runs retrieve → generate has no arrow
*back* to retrieve.

Picture a support bot that only knows one shipping report. A customer asks
"Tell me about the numbers." A human would say "which numbers — shipments,
revenue, or delays?" then search again. The Module 10 chain cannot. It searches
once, maybe gets the wrong chunks, and either answers weakly or refuses.

LangGraph is that "search again" arrow, plus a second arrow "the draft is not
grounded, write it again," plus the pause from notebook 02 when both retries
run out.

We do **not** re-teach hybrid search. We wrap it.


---
# Step 2 of 10 — Import the pipeline. Index the same PDF.

**In one sentence:** add Module 10's folder to `sys.path` and call the classes
you already built.

Jupyter's working directory is sometimes this `notebooks/` folder and sometimes
the repo root. We walk *up* until we find `10_RAG/notebooks/production_rag_chatbot`
and the sample PDF — we do **not** hard-code `Path.cwd().parent.parent`.


In [ ]:
repo_root = next(
    (p for p in [here, *here.parents]
     if (p / "10_RAG" / "notebooks" / "production_rag_chatbot").is_dir()),
    here.parent.parent,
)
RAG_PIPELINE_DIR = repo_root / "10_RAG" / "notebooks" / "production_rag_chatbot"
sys.path.insert(0, str(RAG_PIPELINE_DIR))

from rag_pipeline import (
    HybridIndex, Reranker, load_document, chunk_documents,
    format_sources, ANSWER_PROMPT, CONDENSE_PROMPT,
)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Imported from:", RAG_PIPELINE_DIR)
print("Model        : gpt-4o-mini")


In [ ]:
# Walk until the Module 10 sample PDF exists. Same file Notebook 13 used,
# so retrieval here is comparable to what you already saw.
report_path = next(
    (p / "10_RAG" / "notebooks" / "data" / "sample_report.pdf"
     for p in [here, *here.parents]
     if (p / "10_RAG" / "notebooks" / "data" / "sample_report.pdf").is_file()),
    None,
)
if report_path is None:
    raise FileNotFoundError(
        "Could not find 10_RAG/notebooks/data/sample_report.pdf. "
        "Open this notebook from the course repo, or set report_path yourself."
    )

pages = load_document(str(report_path))
chunks = chunk_documents(pages, chunk_size=500, chunk_overlap=80)

index = HybridIndex()
index.build(chunks)
reranker = Reranker()

print(f"Indexed {len(chunks)} chunks from {report_path}")
print("This is Module 10's HybridIndex + Reranker. We did not reimplement them.")


**What to look at**

- `Imported from:` should end in `production_rag_chatbot`.
- A chunk count, not an error. If the PDF is missing, the raise tells you why.
- Same document as Module 10's capstone, on purpose.


---
# Step 3 of 10 — The whiteboard

**In one sentence:** every node reads and writes the same dictionary. Named
fields are how later nodes know what happened.

Think of a hospital whiteboard at shift change. "Question rewritten twice.
Sources still thin. Draft not grounded." The night nurse does not re-interview
the patient — they read the board.

`messages` uses `add_messages` (notebook 01, Step 4) so chat history **appends**.
The other fields are overwritten each time a node returns them — that is what
we want for `grade` and `answer`.


In [ ]:
from typing import Annotated, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class AgentState(TypedDict, total=False):
    # Chat history. Reducer appends. This is how Turn 4 can see Turn 1.
    messages: Annotated[list, add_messages]
    # The question we actually search with (original, or condensed, or rewritten).
    standalone_question: str
    # Top chunks after hybrid search + rerank. Module 10's objects, not new ones.
    reranked: list
    # "sufficient" | "insufficient" — set by grade_documents.
    grade: str
    # How many times we already rewrote. Cap this or the loop never ends.
    rewrite_count: int
    # The draft the user might see.
    answer: str
    # "grounded" | "not_grounded" | "human_provided" | "refused"
    groundedness: str
    # How many times we already regenerated. Cap = 1 in this notebook.
    regenerate_count: int


# 👇 Caps are product decisions. Two rewrites is "try a bit harder." Twenty is a bill.
MAX_REWRITES = 2
MAX_REGENERATE = 1
print("State fields:", list(AgentState.__annotations__))
print("Caps        : rewrites", MAX_REWRITES, "| regenerates", MAX_REGENERATE)


**What to look at**

| Field | Who writes it | Who reads it |
|---|---|---|
| `messages` | user + `finalize` | `condense`, `generate` (history) |
| `standalone_question` | `condense`, then `rewrite_query` | `retrieve`, `grade_documents` |
| `reranked` | `retrieve` | `grade_documents`, `generate`, `check_groundedness` |
| `grade` | `grade_documents` | `route_after_grade` |
| `rewrite_count` | `condense` (reset 0), `rewrite_query` (+1) | `route_after_grade` vs `MAX_REWRITES` |
| `answer` | `generate` or a human | `check_groundedness`, `finalize` |
| `groundedness` | `check_groundedness` or `human_escalation` | `route_after_groundedness` |
| `regenerate_count` | `condense` (reset 0), `regenerate_bump` (+1) | `route_after_groundedness` vs `MAX_REGENERATE` |

Without the two counters, "try again" becomes an infinite loop. Notebook 01
Step 7 taught the same idea with a critic score.


---
# Step 4 of 10 — Reused nodes: condense + retrieve

**In one sentence:** first-turn questions stay as-is. Follow-ups get rewritten
into a standalone question. Then Module 10 searches.

"What about its risks?" is useless as a search query. `condense` turns it into
"What are the risks or limitations of the total shipments figure?" using
`CONDENSE_PROMPT` — the same prompt Module 10 used, now reading history from
graph state instead of `self.history`.


In [ ]:
def condense(state: AgentState) -> dict:
    # 👇 The latest human message is the raw question for this turn.
    question = state["messages"][-1].content
    history = state["messages"][:-1]
    if not history:
        standalone = question  # first turn: nothing to resolve
    else:
        history_text = "\n".join(f"{m.type}: {m.content}" for m in history[-6:])
        standalone = llm.invoke(
            CONDENSE_PROMPT.format(history=history_text, question=question)
        ).content.strip()
    # Reset retry counters every NEW user turn. Otherwise Turn 2 inherits Turn 1's counts.
    return {"standalone_question": standalone, "rewrite_count": 0, "regenerate_count": 0}


def retrieve(state: AgentState) -> dict:
    # 👇 Identical call shape to ProductionRAGChatbot.chat().
    candidates = index.search(state["standalone_question"], k=8)
    reranked = reranker.rerank(state["standalone_question"], candidates, top_n=4)
    return {"reranked": reranked}


print("condense + retrieve defined. No LangGraph-specific retrieval here.")


**What to look at**

- `condense` returns three keys. It does **not** search.
- `retrieve` does **not** talk to the LLM. It only calls `index` and `reranker`.
- Counters reset here so a later turn cannot be blocked by an earlier retry budget.


---
# Step 5 of 10 — Grade, then rewrite if needed

**In one sentence:** do not trust retrieval. Ask "can these chunks answer this
question?" If no, rewrite the question and search again.

Hiring analogy from Module 06: the bi-encoder is the resume screen. The
cross-encoder is the interview. Grading is the hiring manager asking "did we
even interview the right people?" before you write the offer letter.


In [ ]:
GRADE_PROMPT = """Are the SOURCES below sufficient to answer the QUESTION well? Answer with
exactly one word: SUFFICIENT or INSUFFICIENT.

QUESTION: {question}

SOURCES:
{sources}"""


def grade_documents(state: AgentState) -> dict:
    sources = format_sources(state["reranked"]) if state["reranked"] else "(nothing retrieved)"
    verdict = llm.invoke(
        GRADE_PROMPT.format(question=state["standalone_question"], sources=sources)
    ).content.strip().upper()
    # 👇 Default to sufficient unless the model clearly said INSUFFICIENT.
    # That bias avoids escalating every slightly messy retrieval.
    grade = "insufficient" if "INSUFFICIENT" in verdict else "sufficient"
    return {"grade": grade}


REWRITE_PROMPT = """The QUESTION below did not retrieve sufficient sources from the knowledge
base. Rewrite it to be more specific and retrieval-friendly. Return ONLY the rewritten
question, nothing else.

QUESTION: {question}"""


def rewrite_query(state: AgentState) -> dict:
    rewritten = llm.invoke(
        REWRITE_PROMPT.format(question=state["standalone_question"])
    ).content.strip()
    return {
        "standalone_question": rewritten,
        "rewrite_count": state.get("rewrite_count", 0) + 1,  # 👇 budget used
    }


print("grade_documents + rewrite_query defined.")


The **router** (wired in Step 8) is the actual loop:

```
if grade == sufficient:           go generate
elif rewrite_count < 2:           go rewrite_query → retrieve → grade again
else:                             go human_escalation
```

`rewrite_query` only rewrites. It does not search. The edge
`rewrite_query → retrieve` is what sends us around the loop. That is notebook 01
Step 3 (the missed-search retry), now with an LLM deciding "missed."


---
# Step 6 of 10 — Generate, then check the draft

**In one sentence:** write the answer from the sources, then ask "did we invent
anything?" If yes, write it once more.

Module 10's guardrail scored *retrieval*. This scores the *generated text* —
the same idea as RAGAS Faithfulness (Module 10 Notebook 09), used as a live
gate instead of an offline metric. The production module in Step 10 calls
RAGAS for real. Here we keep a one-word LLM judge so the loop is easy to read.


In [ ]:
def generate(state: AgentState) -> dict:
    # Same ANSWER_PROMPT + format_sources as Module 10. Unmodified.
    history_text = "\n".join(f"{m.type}: {m.content}" for m in state["messages"][:-1][-6:])
    prompt = ANSWER_PROMPT.format(
        sources=format_sources(state["reranked"]),
        history=history_text,
        question=state["standalone_question"],
    )
    answer = llm.invoke(prompt).content.strip()
    return {"answer": answer}


GROUNDEDNESS_PROMPT = """Does the ANSWER rely ONLY on facts present in the SOURCES, with no
invented details? Answer with exactly one word: GROUNDED or NOT_GROUNDED.

SOURCES:
{sources}

ANSWER:
{answer}"""


def check_groundedness(state: AgentState) -> dict:
    verdict = llm.invoke(
        GROUNDEDNESS_PROMPT.format(
            sources=format_sources(state["reranked"]),
            answer=state["answer"],
        )
    ).content.strip().upper()
    groundedness = "not_grounded" if "NOT_GROUNDED" in verdict else "grounded"
    return {"groundedness": groundedness}


def regenerate_bump(state: AgentState) -> dict:
    # A tiny node whose only job is +1. The next edge sends us back to generate.
    # Keeping the increment out of generate() makes the retry budget visible in the trace.
    return {"regenerate_count": state.get("regenerate_count", 0) + 1}


print("generate + check_groundedness + regenerate_bump defined.")


```
if groundedness == grounded:      go finalize
elif regenerate_count < 1:        go regenerate_bump → generate again
else:                             go human_escalation
```

Why a separate `regenerate_bump` node? So the printed trace shows
`[regenerate_bump] {'regenerate_count': 1}` instead of hiding the increment
inside `generate`. You can *see* the budget being spent.


---
# Step 7 of 10 — Ask a human when retries run out

**In one sentence:** same four parts as notebook 02. The *reason* is new:
"we searched twice and the answer is still ungrounded."

No middleware knows that rule. `HumanInTheLoopMiddleware` pauses *before a
named tool*. This pause is *after a custom loop*. That is why we built
`interrupt()` by hand yesterday.


In [ ]:
def human_escalation(state: AgentState) -> dict:
    # 👇 Graph FREEZES. The dict is what a reviewer sees in the Streamlit app.
    guidance = interrupt({
        "reason": "Could not retrieve/generate a grounded answer after retrying.",
        "question": state["standalone_question"],
        "best_effort_answer": state.get("answer"),
        "rewrite_count": state.get("rewrite_count", 0),
        "regenerate_count": state.get("regenerate_count", 0),
    })
    # Command(resume={"human_answer": "..."}) becomes `guidance` here.
    if guidance and guidance.get("human_answer"):
        return {"answer": guidance["human_answer"], "groundedness": "human_provided"}
    return {
        "answer": (
            "I don't have enough grounded information in this document to answer "
            "confidently, and no human guidance was provided."
        ),
        "groundedness": "refused",
    }


def finalize(state: AgentState) -> dict:
    from langchain_core.messages import AIMessage
    # 👇 Append the answer onto messages so the NEXT turn's condense() can see it.
    return {"messages": [AIMessage(content=state["answer"])]}


print("human_escalation + finalize defined.")


**What to look at**

- The payload includes the question, the best-effort draft, and both counters.
  A reviewer should not have to guess why we paused.
- Press Enter (`human_answer` empty) to take the honest-refusal branch. Type
  text only if you want to inject a human answer into the same thread.
- `finalize` is how history grows. Without it, Turn 4 has nothing to condense.


---
# Step 8 of 10 — Wire the graph

**In one sentence:** two routers, two loops, one pause, one checkpointer.


In [ ]:
def route_after_grade(state: AgentState) -> str:
    if state["grade"] == "sufficient":
        return "generate"
    if state.get("rewrite_count", 0) < MAX_REWRITES:
        return "rewrite_query"
    return "human_escalation"


def route_after_groundedness(state: AgentState) -> str:
    if state["groundedness"] == "grounded":
        return "finalize"
    if state.get("regenerate_count", 0) < MAX_REGENERATE:
        return "regenerate_bump"
    return "human_escalation"


builder = StateGraph(AgentState)
for name, fn in [
    ("condense", condense),
    ("retrieve", retrieve),
    ("grade_documents", grade_documents),
    ("rewrite_query", rewrite_query),
    ("generate", generate),
    ("check_groundedness", check_groundedness),
    ("regenerate_bump", regenerate_bump),
    ("human_escalation", human_escalation),
    ("finalize", finalize),
]:
    builder.add_node(name, fn)

builder.add_edge(START, "condense")
builder.add_edge("condense", "retrieve")
builder.add_edge("retrieve", "grade_documents")
builder.add_conditional_edges("grade_documents", route_after_grade, {
    "generate": "generate",
    "rewrite_query": "rewrite_query",
    "human_escalation": "human_escalation",
})
builder.add_edge("rewrite_query", "retrieve")          # 👈 the retrieval loop
builder.add_edge("generate", "check_groundedness")
builder.add_conditional_edges("check_groundedness", route_after_groundedness, {
    "finalize": "finalize",
    "regenerate_bump": "regenerate_bump",
    "human_escalation": "human_escalation",
})
builder.add_edge("regenerate_bump", "generate")        # 👈 the generation loop
builder.add_edge("human_escalation", "finalize")
builder.add_edge("finalize", END)

# No checkpointer → interrupt() cannot resume. Same rule as notebook 02.
agentic_rag = builder.compile(checkpointer=InMemorySaver())
show_graph(agentic_rag)


**What to look at on the drawing**

```
condense → retrieve → grade ── sufficient ──→ generate → groundedness ── grounded ──→ finalize
                 ^                |                         ^                |
                 |           insufficient                   |           not grounded
                 |           (budget left)                  |           (budget left)
                 +────── rewrite_query                      +────── regenerate_bump
                                    \
                                     +── (budget gone, either check) → human_escalation → finalize
```

You should see **cycles**. If the picture is a straight line, an edge is missing.


---
# Step 9 of 10 — Four turns, one thread

We print every node that fired (`stream_mode="updates"`). Assess the **trace**,
not just the final sentence. That is how you debug an agent in a job.

All four turns use `thread_id="capstone-demo"`. Same whiteboard. That is how
Turn 4 can resolve "its."


In [ ]:
from langchain_core.messages import HumanMessage


def ask(question, thread_id="capstone-demo"):
    """Stream one user turn. Returns (config, final_state_or_None).

    If the graph pauses, final_state is None and you must resume with the
    same config (same thread_id). If it finishes, final_state is the board.
    """
    config = {"configurable": {"thread_id": thread_id}}
    print(f"\n=== Q: {question}  (thread={thread_id}) ===")
    paused = False
    for update in agentic_rag.stream(
        {"messages": [HumanMessage(content=question)]},
        config,
        stream_mode="updates",
    ):
        for node_name, node_output in update.items():
            if node_name == "__interrupt__":
                print("  [PAUSED] human_escalation ->", node_output[0].value)
                paused = True
                continue
            shown = {k: v for k, v in node_output.items() if k != "messages"}
            print(f"  [{node_name}] {shown}")
    if paused:
        return config, None
    final_state = agentic_rag.get_state(config).values
    print("ANSWER:", final_state.get("answer"))
    return config, final_state


print("ask() ready. Same thread_id = same checkpointer memory.")


## Turn 1 — a well-scoped question

Expect the happy path: retrieve → sufficient → generate → grounded → finalize.
No rewrite. No pause.


In [ ]:
ask("What were the total shipments mentioned in the report?")


**What to look at**

- Nodes: `condense`, `retrieve`, `grade_documents` (`sufficient`), `generate`,
  `check_groundedness` (`grounded`), `finalize`.
- `rewrite_count` stays 0. No `[PAUSED]`.
- The answer should mention a number from the report, not invent a country.

If this run grades `insufficient` once, that is the live LLM. Read the rewritten
question — you are watching the loop from Step 5 do its job.


## Turn 2 — deliberately vague

"Tell me about the numbers" is how real users talk. A chain searches that
string once. We want to see `rewrite_query` fire, or at least a more specific
`standalone_question` after condense.


In [ ]:
ask("Tell me about the numbers.", thread_id="capstone-demo")


**What to look at**

- Same `thread_id`. `condense` can see Turn 1.
- Either `grade` is `insufficient` then `rewrite_query` prints a sharper
  question, or the model gets lucky and grades sufficient. Both are valid.
- If you see `rewrite_count: 1` or `2`, the retrieval loop ran. That is the
  whole point of this notebook.


## Turn 3 — out of scope, then a human

"What is the capital of France?" is not in a shipping report. We expect retries
to exhaust and `human_escalation` to pause. Then **you** decide: type a grounded
correction, or press Enter so the agent refuses honestly. Do not type Paris
and pretend the document said it — unless you want to show the class what a
bad human override looks like.


In [ ]:
config3, state3 = ask("What is the capital of France?", thread_id="capstone-demo")
print("Paused (state3 is None):", state3 is None)
if config3 is not None:
    snap3 = agentic_rag.get_state(config3)
    print("next node(s) :", snap3.next)
    print("grade        :", snap3.values.get("grade"))
    print("rewrites     :", snap3.values.get("rewrite_count"))
    print("regenerates  :", snap3.values.get("regenerate_count"))


In [ ]:
# Resume ONLY if this run actually paused. A live judge sometimes calls France
# "sufficient" (it knows Paris from pretraining). That is worth seeing — it is
# why production adds a min_rerank_score guardrail in Step 10.
if state3 is None:
    print()
    print("=" * 60)
    print("HUMAN REVIEW — retries ran out. The graph is frozen.")
    print("Type a correction to inject, or press Enter to refuse.")
    print("=" * 60)
    typed = input("  Your answer (blank = refuse): ").strip()
    human_answer = typed if typed else None
    print("→ Injecting human answer." if human_answer else "→ You left it blank. Honest refusal.")
    resumed = agentic_rag.invoke(
        Command(resume={"human_answer": human_answer}),
        config3,
    )
    print("\nResumed answer:", resumed["messages"][-1].content)
else:
    print("This run did not pause — the judge treated France as answerable.")
    print("That is a live-LLM property. The production graph adds a cheap")
    print("rerank-score gate so off-topic questions fail before the LLM judge.")


**What to look at**

- If paused: the cell **waited**. That wait is the same product as notebook 02,
  with a different reason — "retries exhausted," not "risky tool."
- Press Enter: the answer should *not* be "Paris." It should be the refusal.
- Type something: that text becomes the answer (`groundedness: human_provided`).
- If not paused: read the trace. You just saw why a score threshold (no LLM)
  belongs in front of an LLM judge. Step 10's production graph has that gate.


## Turn 4 — a pronoun, to prove memory

"What about its risks or limitations?" only makes sense if `condense` can see
this thread. A new `thread_id` would search the word "its" and fail.


In [ ]:
ask("What about its risks or limitations?", thread_id="capstone-demo")


**What to look at**

- `[condense] {'standalone_question': ...}` should mention shipments (or
  whatever Turn 1–2 were about), not the word "its" alone.
- Same `thread_id`. That is notebook 01 Step 9 (checkpointer), doing real work.

| Product | Happy path | Vague path | Escalation |
|---|---|---|---|
| **Internal wiki Q&A** | "What is our PTO policy?" | "the time-off thing" | "what's the CEO's salary?" not in the wiki |
| **Support** | "Where is order 1842?" | "my package" | "can you also wire me $400?" |
| **This report bot** | Turn 1 | Turn 2 | Turn 3 |


---
# Step 10 of 10 — The production module

The graph above is the teaching version: every node is in this notebook, easy
to read. `capstone_agentic_rag/graph.py` is the **same loops**, plus the rest
of Module 10:

| Addition | Where | Why |
|---|---|---|
| Token-budget trim | `trim_history` | Checkpointer cannot grow forever |
| `Store` preference | `recall_preferences` | "Be concise" survives a **new** thread — Module 10 already had Store; this is where it changes the RAG product |
| `min_rerank_score` | `grade_documents` | Cheap "nothing relevant" gate — the France problem |
| RAGAS Faithfulness | `check_groundedness` | The metric from Module 10 Notebook 09, used live |
| Retry + backoff | `resilient_invoke` | Transient API errors should not kill the turn |
| Structured citations | `extract_structured_citations()` | Ticketing queues need a schema, not prose |

We **import** that module. We do not paste a second copy.


In [ ]:
CAPSTONE_DIR = next(
    (p / "11_LangGraph" / "capstone_agentic_rag"
     for p in [here, *here.parents]
     if (p / "11_LangGraph" / "capstone_agentic_rag" / "graph.py").is_file()),
    Path.cwd().parent / "capstone_agentic_rag",
)
sys.path.insert(0, str(CAPSTONE_DIR))

from graph import build_graph, remember_answer_style, extract_structured_citations

production_graph, production_index, production_reranker, production_store, production_ingest = build_graph()
print("Loaded from :", CAPSTONE_DIR)
print("Nodes       :", list(production_graph.get_graph().nodes.keys()))


## RAGAS faithfulness, scored live

Same shipments question, now through the production graph.
`faithfulness_score` is a number (or `None` if `ragas` is not installed — then
the LLM-judge fallback still sets `groundedness`).


In [ ]:
config_p1 = {"configurable": {"thread_id": "production-demo-1"}}
production_graph.invoke(
    {"messages": [HumanMessage(content="What were the total shipments mentioned in the report?")]},
    config_p1,
)
state_p1 = production_graph.get_state(config_p1).values
print("groundedness          :", state_p1.get("groundedness"))
print("RAGAS faithfulness    :", state_p1.get("faithfulness_score"))
print("answer (first 300 ch) :", (state_p1.get("answer") or "")[:300])


**What to look at**

- `groundedness` should be `grounded` for this in-document question.
- A float between 0 and 1 if RAGAS imported. `None` + a grounded verdict if not.
  The graph must not crash either way.


## Long-term memory on a brand-new thread

A **checkpointer** remembers this conversation. A **`Store`** remembers this
user. Module 10 Notebook 11 already used a Store. Here it changes the product:
write "concise" once, then ask on a thread that never said that word.


In [ ]:
remember_answer_style(production_store, "default-user", "concise")

config_p2 = {"configurable": {"thread_id": "production-demo-2-brand-new"}}
production_graph.invoke(
    {"messages": [HumanMessage(content="What were the total shipments mentioned in the report?")]},
    config_p2,
)
print("This thread never said 'concise'. The Store did.")
print(production_graph.get_state(config_p2).values.get("answer"))


**What to look at**

A short answer (about two sentences). The checkpointer for
`production-demo-2-brand-new` is empty. The preference came from the **Store**,
keyed by `default-user`, not by `thread_id`.

| | Checkpointer (`thread_id`) | Store (user key) |
|---|---|---|
| Remembers | This conversation | This user, any conversation |
| Taught | Notebook 01 Step 9; Turn 4 above ("its") | Module 10 Notebook 11; **this cell** |
| Dies when | Process exits (`InMemorySaver`) | Process exits (`InMemoryStore`) — swap for a DB later |


## Structured citations for a downstream system

The user still sees prose. A ticketing queue wants `{answer, citations: [...]}`.


In [ ]:
structured = extract_structured_citations(
    question=state_p1["standalone_question"],
    answer=state_p1["answer"],
    reranked=state_p1["reranked"],
)
print(type(structured).__name__)
print(structured)


## Tests that do not need an API key

`capstone_agentic_rag/tests/test_graph.py` fakes the retriever and the LLM. It
asserts **which nodes fired**, not what the model said. That is how you CI an
agent: deterministic topology, not flaky wording.

```bash
pytest ../capstone_agentic_rag/tests/test_graph.py -v
```

## Ship it

The Streamlit app at `../capstone_agentic_rag/app.py` shows the live trace,
the answer-style Store control, and the human-approval box when
`human_escalation` pauses.

```bash
cd ../capstone_agentic_rag
pip install -r requirements.txt
streamlit run app.py
```


---
# What you should be able to explain out loud

1. What did we import from Module 10, and what did LangGraph add?
2. Why do `rewrite_count` and `regenerate_count` exist?
3. Why is `human_escalation` a custom `interrupt()`, not
   `HumanInTheLoopMiddleware`?
4. Why must Turn 4 reuse `capstone-demo`?
5. Checkpointer vs Store — which one made the concise answer on a new thread?
   (Checkpointer = this chat. Store = this user, any chat.)

| Primitive (notebooks 01–02) | Used here for |
|---|---|
| Conditional edge + loop | grade → rewrite → retrieve; groundedness → regenerate |
| Caps / stop conditions | `MAX_REWRITES`, `MAX_REGENERATE` |
| `interrupt()` + same `thread_id` | escalate when both loops are spent |
| Checkpointer | multi-turn `condense` |
| `Store` (production module) | answer style across conversations |

**The pattern to reuse on the next product:** take a chain that already works.
Find the one or two places it needs to retry, branch, or wait for a human.
Wrap *those* transitions in a graph. Do not rebuild the pipeline from scratch.
